---
title: Data collection

---

## Data extraction

In [1]:
url = f"https://raw.githubusercontent.com/d-wkim/phd/refs/heads/main/data"

import pandas as pd
import os

os.makedirs(f"../data/", exist_ok = True)

df = pd.read_csv(f"outcomes.csv", encoding = "utf-8")
df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,NaN,NaN,DONE,478,NaN,NaN,Set Lachman and pivot shift at 1+,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,TODO,17,NaN,NaN,https://plotdigitizer.com/app,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,% COMPLETED,96.57%,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,id,NaN,study,download,view,screening,subgroup,data_type,outcome,...,xi,sdi,a,q1,m,q3,b,yi,sei,NaN
4,5.0,1,☑,"Ageberg et al., 2009",https://raw.githubusercontent.com/d-wkim/phd/r...,https://github.com/d-wkim/phd/blob/main/data/p...,Included,patellar,continuous,tegner,...,4.17,1.41,2,NaN,4,NaN,7,NaN,NaN,NaN


In [2]:
def hr():
    from IPython.display import display, HTML
    hr = f"""\
<hr style="height:1.5px; background-color:black; border:None; box-shadow:None;"></hr>\
"""
    display(HTML(hr))

## Data transformation
https://www.math.hkbu.edu.hk/~tongt/papers/median2mean.html

**Table of Contents**\
  **Estimating standard deviation**
  - [proportions to standard deviation](#prop_sd)\
    *Converts events and sample size to proportions, standard error, and standard deviation*.
  - [confidence intervals to standard deviation](#ci_sd)
  - [p-values and effect size to standard deviation](#pv_sd)
  - [range to standard deviation](#range_sd)\
    *Converts minimum and maximum (or range) to standard deviation*.\

**Estimating sample mean from median**
  - [median to mean](#median_mean)

When a study reports median and range, which is more common than reporting mean and standard deviation, most of these studies are excluded from the systematic reviews becauses of the format in which the authors decided to report the results. There are very close approximations, in such a case as when the median, minimum and maximum values are reported by a study. In our study, the required data for analysis was derived by transformation methods instead of excluding the studies, whenever possible.

In [81]:
import pandas as pd
df = pd.DataFrame(columns = ["a", "m", "b", "n"])

For simplicity though, I will use this equation only for studies with a sample size greater than or equal to 70.

<a id="p_sd"></a>

<hr style="height:1.5px; background-color:black; border:None; box-shadow:None;"></hr>

**`p_sd`**: *python function for converting sample size, `n` and number of events, `k` into standard deviation. Returns proportions, standard error, and standard deviation.*

In [3]:
n = 16
k = 4

def prop_sd(k, n):
    import numpy as np
    p = (k + 0.5)/(n + 0.5)
    se = p/(1-p)
    sd = se * np.sqrt(n)
    p, se, sd = [round(x, 3) for x in (p, se, sd)]
    print(f"""\
p  = {p}
se = {se}
sd = {sd} \
""")
    return sd

In [4]:
sd = prop_sd(4, 16)

p  = 0.273
se = 0.375
sd = 1.5 


In [61]:
print(sd)

1.5


## Range to SD

<hr style="height:1.5px; background-color:black; border:None; box-shadow:None;"></hr>

https://www.math.hkbu.edu.hk/~tongt/papers/median2mean.html

$$
\hat{\sigma}
=
\frac{b-a}
{2\Phi^{-1}\!\left(\frac{n-0.375}{n+0.25}\right)}
$$

In [83]:
n = 38
mean = 1.8
sd = 2.2
a = -3
b = 5

In [2]:
from scipy.stats import norm

def range_sd(n, a, b):
    range = b - a
    z = ((n-0.375)/(n+0.25))
    sd = 2 * norm.ppf(z)
    sd = round(sd, 3)
    print(sd)
    return sd

In [123]:
def hozo(n, a, b):
    range = b - a
    if n <= 15:
        sd = range/4
    if n >= 15 and n < 70:
        sd = range/6
    sd = round(sd, 3)
    print(sd)
    return sd

In [127]:
SD = range_sd(38, -3, 5)
Wan = range_sd(38, -3, 5)
Hozo = hozo(38, -3, 5)

4.272
4.272
1.333


## Confidence Intervals to SD

In [2]:
def ci_sd(upper, lower, n):
    import numpy as np
    se = (upper - lower)/3.92
    sd = se * np.sqrt(n)
    print(round(sd, 3))

In [4]:
ci_sd(1.6, 0.7, 54)
ci_sd(2, 1.2, 58)

1.687
1.554


In [15]:
ci_sd(96.1, 84.4, 23)

14.314


There are two ways to derive SD. (1) From CIs and (2) from p-values.
In Taylor et al., 2009 method 1 resulted in 13.8 and 14.3, respectively, while method 2 resulted in 12.19 and 12.75, respectively for the two subgroups. The lesser values were recorded.

## P-values to SD

In [16]:
def pv_sd(pv, xi, n): # xi is effect size estimate
    from scipy.stats import norm
    import numpy as np
    z = abs(norm.ppf(pv/2))
    se = xi/z
    sd = se * np.sqrt(n)
    print(round(sd, 3))

Example: **Taylor et al., 2009**

**Lysholm**

In [17]:
pv = 0.97
xi = 90.4 - 90.3
n = 21
pv_sd(pv, xi, n = 21) # BPTB
pv_sd(pv, xi, n = 23) # HT

12.185
12.752


**Tegner**

In [18]:
# BPTB
pv = 0.04
xi = 6.8 - 5.3
n = 20
pv_sd(pv, xi, n)

3.266


In [19]:
# HST
pv = 0.04
xi = 6.8 - 5.3
n = 24
pv_sd(pv, xi, n)

3.578


# Estimating sample mean from median

(1) if the range is given alongside the median, use Hozo et al., 2010 method:

In [22]:
def Hozo(min, median, max, n):
    if n <= 25:
        mean = (min + (2*median) + max)/4
    else:
        mean = median
    return mean

(2) if the interquartile range (IQR) is given alongside the median, use Wan et al., 2014 method:

In [28]:
def Wan(q1, median, q3):
    mean = (q1 + median + q3)/3
    return mean

(3) if the range ***and*** interquartile range are given alongside the median, use Luo et al., 2018 method:

In [27]:
def Luo(min, q1, median, q3, max, n):
    w1 = (2.2/(2.2 + (n^0.75)))
    w2 = (0.7 - (0.72 / (n^0.55)))
    mean = (w1 * ((min + max)/2)) + (w2 * ((q1 + q3)/2)) + (1 - w1 - w2) * median
    return mean

In [31]:
Wan(87, 90, 93)

90.0

In [32]:
Wan(90, 98, 100)

96.0

In [33]:
Wan(83, 90, 91.5)

88.16666666666667

In [34]:
Wan(91, 95, 100)

95.33333333333333